# 03 — Model Inference and API Smoke Test

This notebook covers inference and demonstrates how the saved model artifact is reused. It can also send a sample request to a locally running FastAPI service.

**Important:** Inputs below are illustrative examples, not medical advice or clinically validated values.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from heart_disease.predict import load_model, predict_one, MODEL_PATH
print("Expected model artifact:", MODEL_PATH)


In [ ]:
# Train first if the model artifact is not available.
if not MODEL_PATH.exists():
    print("No trained model found. Run notebook 02 or `python -m heart_disease.train` first.")
else:
    model = load_model()
    print("Model loaded:", type(model).__name__)


In [ ]:
# Example feature payload matching the UCI Cleveland-style schema.
sample_patient = {
    "age": 63, "sex": 1, "cp": 3, "trestbps": 145, "chol": 233,
    "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0,
    "oldpeak": 2.3, "slope": 0, "ca": 0, "thal": 1
}

if MODEL_PATH.exists():
    result = predict_one(sample_patient, model=model)
    result


## Test the running API

Start the API in a separate terminal from the project root:

```bash
uvicorn heart_disease.api:app --host 127.0.0.1 --port 8000
```

Then run the cell below. If the API is not running, the request will fail; that is expected until you start the service.


In [ ]:
import requests

API_URL = "http://127.0.0.1:8000"
try:
    health_response = requests.get(f"{API_URL}/health", timeout=3)
    print("Health status:", health_response.status_code, health_response.json())
    predict_response = requests.post(f"{API_URL}/predict", json=sample_patient, timeout=10)
    print("Prediction status:", predict_response.status_code)
    display(predict_response.json())
except requests.RequestException as exc:
    print("API not reachable. Start uvicorn in a separate terminal.")
    print(type(exc).__name__, str(exc))


## Reproducibility and safety notes

- Save the model pipeline, not just the estimator, so inference applies the same preprocessing.
- Validate request schemas and log failures without logging unnecessary personal data.
- Treat model confidence as a model score, not a guaranteed probability of disease.
- This project is educational only and must not be used for diagnosis or treatment decisions.
